# GLOBEM Feature-Level Generalization Check — Multi-Year
## Component 2 — Anxiety Vulnerability Mapping (R26-DS-012)
### Runs across all downloaded GLOBEM years (INS-W_1 .. INS-W_4)

---
**SCOPE — read before quoting any number from this notebook:**

This notebook is **not** an external validation of the trained GATv2 model
or its behavioral-graph architecture from `digital_phenotyping_v14_polished.ipynb`.

GLOBEM's public credentialed release provides only RAPIDS-derived,
pre-aggregated **daily** sensor features — no raw GPS pings — so the
stay-point / contextual-state graph construction used for StudentLife
cannot be reproduced here. There is nothing to fix about this; it's a
data-availability limit of GLOBEM's public release, not a pipeline bug.

**What this notebook does instead:** trains a fresh tabular classifier
on GLOBEM's own aggregate location + EMA features to test whether
passive-sensing-derived anxiety signal generalizes:
1. **Within-cohort** (grouped, repeated CV, per year and pooled)
2. **Across cohorts/years** (leave-one-year-out — train on N-1 years,
   test on a fully held-out year/cohort — the closest thing to a true
   external validation GLOBEM's public data supports, since each
   INS-W_n year is a separately recruited cohort, collected in a
   different period, e.g. Year 3 = COVID onset, Year 4 = recovery)

Reported with the same 4-number honest discipline as Component 2:
main AUC, behavioral-only (no EMA) AUC, permutation AUC, honest range.


In [4]:
# ============================================================
# CELL 1: Install libraries, mount Drive, imports
# ============================================================
!pip install scikit-learn pandas numpy -q

from google.colab import drive
drive.mount('/content/drive')

import os, warnings
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("\u2705 Libraries loaded")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Libraries loaded


In [5]:
# ============================================================
# CELL 2: Configuration
# ============================================================
# Root folder containing INS-W_1 .. INS-W_4 subfolders, each with
# FeatureData/location.csv and SurveyData/{ema.csv, dep_weekly.csv}
GLOBEM_ROOT = "/content/drive/MyDrive/Anxiety/globem-dataset-multi-year-datasets-for-longitudinal-human-behavior-modeling-generalization-1.1/globem-dataset-multi-year-datasets-for-longitudinal-human-behavior-modeling-generalization-1.1/"

# Which years to attempt to load — script skips any year whose
# folder isn't present, so this is safe to leave as-is even if you
# haven't downloaded every year yet.
YEARS = ["INS-W_1", "INS-W_2", "INS-W_3", "INS-W_4"]

OUTPUT_DIR = "/content/drive/MyDrive/Anxiety/globem_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# RAPIDS feature window to use for location features.
# ':7dhist' = trailing 7-day aggregate as of that date — naturally
# aligned with dep_weekly's weekly assessment cadence.
FEATURE_WINDOW_SUFFIX = ":7dhist"

N_SPLITS  = 5   # grouped stratified k-fold
N_REPEATS = 5   # repeats for stabilised AUC estimate (same approach as your 3-fold/5-repeat)

print("\u2705 Configuration set")
print(f"Looking for years: {YEARS}")


✅ Configuration set
Looking for years: ['INS-W_1', 'INS-W_2', 'INS-W_3', 'INS-W_4']


In [6]:
# ============================================================
# CELL 3: Per-year data loading
# ============================================================
def load_year(year):
    # Load location, ema, dep_weekly for one GLOBEM year folder.
    # Returns None if the year folder / required files are missing.
    base = os.path.join(GLOBEM_ROOT, year)
    loc_path = os.path.join(base, "FeatureData", "location.csv")
    ema_path = os.path.join(base, "SurveyData", "ema.csv")
    dep_path = os.path.join(base, "SurveyData", "dep_weekly.csv")

    if not (os.path.exists(loc_path) and os.path.exists(dep_path)):
        print(f"  \u26a0 {year}: missing location.csv or dep_weekly.csv \u2014 skipped")
        return None

    loc = pd.read_csv(loc_path)
    loc["date"] = pd.to_datetime(loc["date"])

    dep = pd.read_csv(dep_path)
    dep["date"] = pd.to_datetime(dep["date"])
    dep = dep.dropna(subset=["anx_weekly_subscale"]).copy()
    dep["label"] = dep["anx_weekly_subscale"].astype(bool).astype(int)

    if os.path.exists(ema_path):
        ema = pd.read_csv(ema_path)
        ema["date"] = pd.to_datetime(ema["date"])
    else:
        ema = pd.DataFrame(columns=["pid", "date", "negative_affect_EMA"])

    return {"location": loc, "ema": ema, "dep_weekly": dep}

print("Loading GLOBEM years...")
raw_data = {}
for yr in YEARS:
    result = load_year(yr)
    if result is not None:
        raw_data[yr] = result
        print(f"  \u2705 {yr}: {result['location'].pid.nunique()} participants "
              f"in location.csv, {len(result['dep_weekly'])} labelled dep_weekly rows")

if not raw_data:
    raise RuntimeError(
        "No GLOBEM years found under GLOBEM_ROOT \u2014 check the path in CELL 2 "
        "matches your Google Drive folder structure."
    )

print(f"\n\u2705 Loaded {len(raw_data)} year(s): {list(raw_data.keys())}")


Loading GLOBEM years...
  ✅ INS-W_1: 155 participants in location.csv, 2221 labelled dep_weekly rows
  ✅ INS-W_2: 218 participants in location.csv, 2046 labelled dep_weekly rows
  ✅ INS-W_3: 137 participants in location.csv, 1318 labelled dep_weekly rows
  ✅ INS-W_4: 195 participants in location.csv, 1977 labelled dep_weekly rows

✅ Loaded 4 year(s): ['INS-W_1', 'INS-W_2', 'INS-W_3', 'INS-W_4']


In [7]:
# ============================================================
# CELL 4: Build per-year merged feature tables, then concatenate
# ============================================================
def trailing_ema_features(row, ema_df):
    window = ema_df[(ema_df.pid == row.pid) &
                     (ema_df.date <= row.date) &
                     (ema_df.date > row.date - pd.Timedelta(days=7))]
    return pd.Series({
        "ema_mean_7d":  window["negative_affect_EMA"].mean(),
        "ema_count_7d": len(window),
    })

def build_year_table(year, data):
    loc, ema, dep = data["location"], data["ema"], data["dep_weekly"]

    # Continuous 7-day-history location features only.
    # '_dis:' columns are categorical low/med/high discretizations of
    # the same underlying signal \u2014 dropped to keep a clean numeric matrix.
    loc_feat_cols = [c for c in loc.columns
                     if c.endswith(FEATURE_WINDOW_SUFFIX) and "_dis:" not in c]

    merged = dep.merge(loc[["pid", "date"] + loc_feat_cols],
                        on=["pid", "date"], how="inner")

    if len(merged) == 0:
        return None, loc_feat_cols

    ema_feats = merged.apply(trailing_ema_features, axis=1, ema_df=ema)
    merged = pd.concat([merged, ema_feats], axis=1)
    merged["year"] = year
    # Keep participant IDs distinguishable across years UNLESS you know
    # for certain the same pid = same physical person across years in
    # your downloaded release. GLOBEM's docs mention cross-year same-user
    # tracking is possible in principle \u2014 if so, drop this line so
    # grouping is done on raw pid instead of pid+year.
    merged["group_id"] = merged["pid"] + "_" + year

    return merged, loc_feat_cols

print("Building per-year feature tables...")
year_tables = {}
common_loc_cols = None
for yr, data in raw_data.items():
    tbl, loc_cols = build_year_table(yr, data)
    if tbl is None:
        print(f"  \u26a0 {yr}: no overlapping location+label rows \u2014 skipped")
        continue
    year_tables[yr] = tbl
    common_loc_cols = set(loc_cols) if common_loc_cols is None else (common_loc_cols & set(loc_cols))
    print(f"  \u2705 {yr}: {len(tbl)} participant-weeks, {tbl.pid.nunique()} participants, "
          f"{tbl['label'].mean():.1%} anxious-week")

common_loc_cols = sorted(common_loc_cols)   # feature columns present in every year
print(f"\n\u2705 {len(common_loc_cols)} location features common across all loaded years")

# Concatenate all years into one master table, restricted to the
# common feature columns so cross-year models are apples-to-apples.
EMA_COLS = ["ema_mean_7d", "ema_count_7d"]
ALL_COLS = common_loc_cols + EMA_COLS

master = pd.concat([t[["pid", "year", "group_id", "label"] + ALL_COLS]
                     for t in year_tables.values()], ignore_index=True)
print(f"\nMaster table: {len(master)} participant-weeks across {master.year.nunique()} year(s)")
master.to_csv(OUTPUT_DIR + "globem_master_features.csv", index=False)


Building per-year feature tables...
  ✅ INS-W_1: 2221 participant-weeks, 154 participants, 26.7% anxious-week
  ✅ INS-W_2: 2046 participant-weeks, 218 participants, 24.4% anxious-week
  ✅ INS-W_3: 1219 participant-weeks, 136 participants, 17.6% anxious-week
  ✅ INS-W_4: 1977 participant-weeks, 195 participants, 20.3% anxious-week

✅ 82 location features common across all loaded years

Master table: 7463 participant-weeks across 4 year(s)


In [8]:
# ============================================================
# CELL 5: Evaluation helper \u2014 grouped, repeated Stratified-K-Fold
# ============================================================
imputer_template = SimpleImputer(strategy="median")

def evaluate(X, y, groups, label, n_splits=N_SPLITS, n_repeats=N_REPEATS, verbose=True):
    aucs, f1s, precs, recs = [], [], [], []
    for rep in range(n_repeats):
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True,
                                     random_state=RANDOM_STATE + rep)
        for train_idx, test_idx in sgkf.split(X, y, groups):
            imp = SimpleImputer(strategy="median")
            X_train = imp.fit_transform(X[train_idx])
            X_test  = imp.transform(X[test_idx])
            clf = RandomForestClassifier(
                n_estimators=300, max_depth=6, min_samples_leaf=5,
                class_weight="balanced", random_state=RANDOM_STATE
            )
            clf.fit(X_train, y[train_idx])
            proba = clf.predict_proba(X_test)[:, 1]
            pred  = (proba >= 0.5).astype(int)
            aucs.append(roc_auc_score(y[test_idx], proba))
            f1s.append(f1_score(y[test_idx], pred, zero_division=0))
            precs.append(precision_score(y[test_idx], pred, zero_division=0))
            recs.append(recall_score(y[test_idx], pred, zero_division=0))
    if verbose:
        print(f"\n-- {label} --")
        print(f"  AUC  : {np.mean(aucs):.3f} \u00b1 {np.std(aucs):.3f}")
        print(f"  F1   : {np.mean(f1s):.3f}")
        print(f"  Prec : {np.mean(precs):.3f}")
        print(f"  Rec  : {np.mean(recs):.3f}")
    return np.mean(aucs), np.std(aucs)

print("\u2705 Evaluation helper ready")


✅ Evaluation helper ready


In [9]:
# ============================================================
# CELL 6: Within-cohort results \u2014 per year, then pooled across years
# ============================================================
results_within = []

for yr in year_tables:
    sub = master[master.year == yr]
    X_all = sub[ALL_COLS].values
    X_beh = sub[common_loc_cols].values
    y     = sub["label"].values
    groups = sub["pid"].values

    print("\n" + "=" * 60)
    print(f"  YEAR: {yr}  (n={len(sub)} weeks, {sub.pid.nunique()} participants)")
    print("=" * 60)
    auc_main, _ = evaluate(X_all, y, groups, f"{yr}: location+EMA")
    auc_beh,  _ = evaluate(X_beh, y, groups, f"{yr}: location-only (no EMA)")

    y_shuf = y.copy()
    np.random.RandomState(RANDOM_STATE).shuffle(y_shuf)
    auc_perm, _ = evaluate(X_all, y_shuf, groups, f"{yr}: permutation test", verbose=False)

    results_within.append({
        "year": yr, "n_weeks": len(sub), "n_participants": sub.pid.nunique(),
        "auc_main": auc_main, "auc_behavioral_only": auc_beh, "auc_permutation": auc_perm
    })

# Pooled across all loaded years
X_all = master[ALL_COLS].values
X_beh = master[common_loc_cols].values
y     = master["label"].values
groups = master["pid"].values   # see note in CELL 4 re: pid vs pid+year grouping

print("\n" + "=" * 60)
print(f"  POOLED ACROSS ALL YEARS (n={len(master)} weeks)")
print("=" * 60)
auc_main_pooled, _ = evaluate(X_all, y, groups, "Pooled: location+EMA")
auc_beh_pooled,  _ = evaluate(X_beh, y, groups, "Pooled: location-only (no EMA)")
y_shuf = y.copy()
np.random.RandomState(RANDOM_STATE).shuffle(y_shuf)
auc_perm_pooled, _ = evaluate(X_all, y_shuf, groups, "Pooled: permutation test", verbose=False)

results_within_df = pd.DataFrame(results_within)
results_within_df.to_csv(OUTPUT_DIR + "globem_within_cohort_results.csv", index=False)
print("\n\u2705 Within-cohort results saved")



  YEAR: INS-W_1  (n=2221 weeks, 154 participants)

-- INS-W_1: location+EMA --
  AUC  : 0.846 ± 0.024
  F1   : 0.655
  Prec : 0.592
  Rec  : 0.745

-- INS-W_1: location-only (no EMA) --
  AUC  : 0.541 ± 0.039
  F1   : 0.241
  Prec : 0.294
  Rec  : 0.211

  YEAR: INS-W_2  (n=2046 weeks, 218 participants)

-- INS-W_2: location+EMA --
  AUC  : 0.756 ± 0.046
  F1   : 0.522
  Prec : 0.501
  Rec  : 0.550

-- INS-W_2: location-only (no EMA) --
  AUC  : 0.510 ± 0.031
  F1   : 0.215
  Prec : 0.244
  Rec  : 0.200

  YEAR: INS-W_3  (n=1219 weeks, 136 participants)

-- INS-W_3: location+EMA --
  AUC  : 0.687 ± 0.075
  F1   : 0.328
  Prec : 0.350
  Rec  : 0.330

-- INS-W_3: location-only (no EMA) --
  AUC  : 0.559 ± 0.080
  F1   : 0.184
  Prec : 0.225
  Rec  : 0.177

  YEAR: INS-W_4  (n=1977 weeks, 195 participants)

-- INS-W_4: location+EMA --
  AUC  : 0.776 ± 0.054
  F1   : 0.480
  Prec : 0.451
  Rec  : 0.525

-- INS-W_4: location-only (no EMA) --
  AUC  : 0.550 ± 0.078
  F1   : 0.234
  Prec : 0

In [10]:
# ============================================================
# CELL 7: Cross-cohort external validation \u2014 leave-one-year-out
# ============================================================
# This is the closest thing to a genuine "external validation" that
# GLOBEM's public data supports: each INS-W_n year is a SEPARATELY
# RECRUITED cohort collected in a different period (Year 3 = COVID
# onset 2020, Year 4 = recovery 2021), so training on N-1 years and
# testing on a fully held-out year tests generalization across cohort
# AND across time \u2014 a stronger claim than same-year CV.
#
# Only runs if you have 2+ years loaded.

def evaluate_holdout(X_train, y_train, X_test, y_test, label):
    imp = SimpleImputer(strategy="median")
    X_train_i = imp.fit_transform(X_train)
    X_test_i  = imp.transform(X_test)
    clf = RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=5,
        class_weight="balanced", random_state=RANDOM_STATE
    )
    clf.fit(X_train_i, y_train)
    proba = clf.predict_proba(X_test_i)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    auc  = roc_auc_score(y_test, proba)
    f1   = f1_score(y_test, pred, zero_division=0)
    print(f"  {label:<45} AUC={auc:.3f}  F1={f1:.3f}  (train n={len(y_train)}, test n={len(y_test)})")
    return auc, f1

results_loyo = []

if len(year_tables) >= 2:
    print("=" * 60)
    print("  LEAVE-ONE-YEAR-OUT CROSS-COHORT VALIDATION")
    print("=" * 60)
    for held_out_year in year_tables:
        train_mask = master.year != held_out_year
        test_mask  = master.year == held_out_year

        for feat_cols, tag in [(ALL_COLS, "location+EMA"), (common_loc_cols, "location-only")]:
            auc, f1 = evaluate_holdout(
                master.loc[train_mask, feat_cols].values, master.loc[train_mask, "label"].values,
                master.loc[test_mask,  feat_cols].values, master.loc[test_mask,  "label"].values,
                f"Train=all-but-{held_out_year}  Test={held_out_year}  [{tag}]"
            )
            results_loyo.append({
                "held_out_year": held_out_year, "feature_set": tag,
                "auc": auc, "f1": f1,
                "n_train": int(train_mask.sum()), "n_test": int(test_mask.sum())
            })
else:
    print("Only 1 year loaded \u2014 leave-one-year-out needs 2+ years. "
          "Download additional INS-W_n years and re-run from CELL 3 to enable this.")

results_loyo_df = pd.DataFrame(results_loyo)
if len(results_loyo_df):
    results_loyo_df.to_csv(OUTPUT_DIR + "globem_leave_one_year_out_results.csv", index=False)
    print("\n\u2705 Leave-one-year-out results saved")


  LEAVE-ONE-YEAR-OUT CROSS-COHORT VALIDATION
  Train=all-but-INS-W_1  Test=INS-W_1  [location+EMA] AUC=0.837  F1=0.644  (train n=5242, test n=2221)
  Train=all-but-INS-W_1  Test=INS-W_1  [location-only] AUC=0.548  F1=0.379  (train n=5242, test n=2221)
  Train=all-but-INS-W_2  Test=INS-W_2  [location+EMA] AUC=0.787  F1=0.593  (train n=5417, test n=2046)
  Train=all-but-INS-W_2  Test=INS-W_2  [location-only] AUC=0.563  F1=0.399  (train n=5417, test n=2046)
  Train=all-but-INS-W_3  Test=INS-W_3  [location+EMA] AUC=0.763  F1=0.485  (train n=6244, test n=1219)
  Train=all-but-INS-W_3  Test=INS-W_3  [location-only] AUC=0.596  F1=0.227  (train n=6244, test n=1219)
  Train=all-but-INS-W_4  Test=INS-W_4  [location+EMA] AUC=0.774  F1=0.423  (train n=5486, test n=1977)
  Train=all-but-INS-W_4  Test=INS-W_4  [location-only] AUC=0.538  F1=0.075  (train n=5486, test n=1977)

✅ Leave-one-year-out results saved


In [11]:
# ============================================================
# CELL 8: Feature importance (pooled model, for interpretation)
# ============================================================
imp = SimpleImputer(strategy="median")
X_full = imp.fit_transform(master[ALL_COLS].values)
clf_full = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=5,
    class_weight="balanced", random_state=RANDOM_STATE
)
clf_full.fit(X_full, master["label"].values)

importances = pd.Series(clf_full.feature_importances_, index=ALL_COLS).sort_values(ascending=False)
print("Top 15 features (pooled model):")
print(importances.head(15))

importances.to_csv(OUTPUT_DIR + "globem_feature_importances.csv")
print("\n\u2705 Feature importances saved")


Top 15 features (pooled model):
ema_mean_7d                                                        0.497097
f_loc:phone_locations_barnett_circdnrtn:7dhist                     0.021655
f_loc:phone_locations_barnett_wkenddayrtn:7dhist                   0.016958
f_loc:phone_locations_locmap_duration_in_locmap_exercise:7dhist    0.014882
f_loc:phone_locations_locmap_percent_in_locmap_greens:7dhist       0.014778
f_loc:phone_locations_locmap_duration_in_locmap_greens:7dhist      0.014104
f_loc:phone_locations_locmap_percent_in_locmap_exercise:7dhist     0.012684
f_loc:phone_locations_doryab_movingtostaticratio:7dhist            0.012355
f_loc:phone_locations_doryab_locationvariance_norm:7dhist          0.008940
f_loc:phone_locations_barnett_avgflightdur:7dhist                  0.008488
f_loc:phone_locations_doryab_radiusgyration_norm:7dhist            0.008432
f_loc:phone_locations_doryab_normalizedlocationentropy:7dhist      0.008346
f_loc:phone_locations_doryab_avgspeed:7dhist            

In [12]:
# ============================================================
# CELL 9: Summary \u2014 same 4-number honest discipline as Component 2
# ============================================================
print("=" * 66)
print("  GLOBEM MULTI-YEAR GENERALIZATION CHECK \u2014 FINAL SUMMARY")
print("=" * 66)
print(f"  Years loaded        : {list(year_tables.keys())}")
print(f"  Total participants  : {master.pid.nunique()}")
print(f"  Total part.-weeks   : {len(master)}")
print(f"  Common loc features : {len(common_loc_cols)}")
print()
print("  -- WITHIN-COHORT (pooled, grouped repeated CV) ------------")
print(f"  Main AUC (loc+EMA)      : {auc_main_pooled:.3f}")
print(f"  Behavioral-only AUC     : {auc_beh_pooled:.3f}")
print(f"  Permutation AUC         : {auc_perm_pooled:.3f}  (validity check)")
print(f"  Honest range            : [{min(auc_main_pooled, auc_beh_pooled):.3f}, "
      f"{max(auc_main_pooled, auc_beh_pooled):.3f}]")
print()
if len(results_loyo_df):
    print("  -- CROSS-COHORT (leave-one-year-out) -----------------------")
    for _, r in results_loyo_df.iterrows():
        print(f"  Held out {r['held_out_year']:<10} [{r['feature_set']:<14}] AUC={r['auc']:.3f}")
print()
print("  SCOPE REMINDER: these numbers characterize generalization of")
print("  behavioral-SENSING SIGNAL on GLOBEM's own aggregate features.")
print("  They are NOT a validation of the trained GATv2 model/graph")
print("  architecture from digital_phenotyping_v14_polished.ipynb \u2014")
print("  report them as a separate, clearly-labeled analysis.")
print(f"\nAll result files written to: {OUTPUT_DIR}")


  GLOBEM MULTI-YEAR GENERALIZATION CHECK — FINAL SUMMARY
  Years loaded        : ['INS-W_1', 'INS-W_2', 'INS-W_3', 'INS-W_4']
  Total participants  : 703
  Total part.-weeks   : 7463
  Common loc features : 82

  -- WITHIN-COHORT (pooled, grouped repeated CV) ------------
  Main AUC (loc+EMA)      : 0.802
  Behavioral-only AUC     : 0.569
  Permutation AUC         : 0.499  (validity check)
  Honest range            : [0.569, 0.802]

  -- CROSS-COHORT (leave-one-year-out) -----------------------
  Held out INS-W_1    [location+EMA  ] AUC=0.837
  Held out INS-W_1    [location-only ] AUC=0.548
  Held out INS-W_2    [location+EMA  ] AUC=0.787
  Held out INS-W_2    [location-only ] AUC=0.563
  Held out INS-W_3    [location+EMA  ] AUC=0.763
  Held out INS-W_3    [location-only ] AUC=0.596
  Held out INS-W_4    [location+EMA  ] AUC=0.774
  Held out INS-W_4    [location-only ] AUC=0.538

  SCOPE REMINDER: these numbers characterize generalization of
  behavioral-SENSING SIGNAL on GLOBEM's own 